In [71]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import ee
import geemap
from utils.variables import (
    PROJECT,
    PAS_ASSET_ID,
    OECMS_ASSET_ID,
    PSM_CRS,
    RAND_SEED,
    PSM_CELL_SIZE,
    EXTERIOR_CELLS_TEST,
)

import geopandas as gpd
from shapely.geometry import box

import os
os.chdir("/Users/alanalutz/Documents/GitHub/tpae/")


ee.Authenticate()
ee.Initialize(project=PROJECT)

In [72]:
# Get all PAs
PAs = ee.FeatureCollection(PAS_ASSET_ID)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID)

all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)

# Get test PA
site_id = 7949
test_site = ee.Feature(all_PAs.filter(ee.Filter.eq("SITE_ID", site_id)).first())

In [73]:
# Create a 10-50km buffer zone around PA
buffer_50 = test_site.buffer(50000)
buffer_10 = test_site.buffer(10000)
donut = buffer_50.difference(buffer_10)

donut_PAs = all_PAs.filterBounds(donut.geometry())

unprotected_donut = donut.difference(donut_PAs.geometry())

In [74]:
# Sample random points within unprotected donut
points = (
    ee.Image.constant(site_id)
    .rename("WDPA_PID")
    .sample(
        region=unprotected_donut.geometry(),
        scale=3000, # approximation of 3km min distance between points
        numPixels=100,
        seed=RAND_SEED,
        geometries=True
    )
)

In [75]:
from pathlib import Path
Path(EXTERIOR_CELLS_TEST).resolve()

PosixPath('/Users/alanalutz/Documents/GitHub/tpae/data/exterior_cells_test.parquet')

In [76]:
# Convert points to GeoDataFrame
points_gdf = gpd.GeoDataFrame.from_features(points.getInfo()["features"], crs="EPSG:4326")

# Reproject to meter-based CRS for 1km x 1km box construction
points_gdf = points_gdf.to_crs(epsg=PSM_CRS)

cell_size = PSM_CELL_SIZE
half = cell_size / 2.0

cells = []
WDPA_PIDs = []
for _, row in points_gdf.iterrows():
    x, y = row.geometry.x, row.geometry.y
    cell_geom = box(x - half, y - half, x + half, y + half)
    cells.append(cell_geom)
    WDPA_PIDs.append(row["WDPA_PID"])

cells = gpd.GeoDataFrame({"geometry": cells, "WDPA_PID": WDPA_PIDs}, crs=points_gdf.crs)
cells["geometry"] = cells.geometry.set_precision(1.0)
cells = cells.drop_duplicates(subset="geometry")
cells["protected"] = 0

cells = cells.to_crs(epsg=4326)

cells.head()
cells.to_parquet(EXTERIOR_CELLS_TEST)


In [73]:
# Visualization

Map = geemap.Map()

Map.addLayer(test_site, {}, "Test PA")
Map.addLayer(unprotected_donut, {}, "Unprotected Donut")
Map.addLayer(points, {}, "Points")
# Map.addLayer(all_PAs, {}, "All PAs")
Map.centerObject(test_site)

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…